In [18]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

## state define

In [19]:
class State(TypedDict):
    question: str
    research: str
    summary: str

In [20]:
!pip install langchain_ollama


[notice] A new release of pip is available: 25.3 -> 26.2
[notice] To update, run: pip install --upgrade pip


## create LLM

In [21]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.2:latest", temperature=0.0)

## Research Node

In [22]:
def research_node(state: State):

    question = state["question"]

    response = llm.invoke(
        f"""
        Research the following topic.

        Topic:
        {question}

        Write detailed research notes.
        """
    )

    state["research"] = response.content

    return state

## Summary Node

In [23]:
def summary_node(state: State):

    notes = state["research"]

    response = llm.invoke(
        f"""
        Summarize the following research notes.

        Notes:
        {notes}
        """
    )

    state["summary"] = response.content
    return state

In [26]:
graph = StateGraph(State)

graph.add_node("research", research_node)
graph.add_node("summary", summary_node)

graph.add_edge(START, "research")
graph.add_edge("research", "summary")
graph.add_edge("summary", END)

agent = graph.compile()

In [27]:
result = agent.invoke(
    {
        "question": "What is LangGraph?",
        "research": "",
        "summary": "",
    }
)

In [28]:
print(result)

{'question': 'What is LangGraph?', 'research': "**Research Notes: LangGraph**\n\nLangGraph is an open-source, graph-based neural network framework designed to facilitate the development of large-scale language models and other deep learning applications. It was created by Facebook AI Research (FAIR) in 2019.\n\n**Key Features:**\n\n1. **Modular Architecture**: LangGraph is built around a modular architecture that allows developers to easily compose and reuse components, making it easier to build complex models.\n2. **Graph-Based Neural Networks**: LangGraph uses graph-based neural networks, which are particularly well-suited for modeling complex relationships between entities in language data.\n3. **Efficient Inference**: LangGraph is designed with efficient inference in mind, using techniques such as sparse matrix operations and optimized memory management to reduce computational overhead.\n4. **Scalability**: LangGraph is designed to scale to large models and datasets, making it suit